## Review dashboard

https://www.eia.gov/electricity/gridmonitor/dashboard/electric_overview/US48/US48

## Get desired data endpoint from API

https://www.eia.gov/opendata/browser/electricity/rto/fuel-type-data

## Initialize route in Python

In [1]:
from eia import EIAClient

client = EIAClient() 
route = client.get_data_endpoint('electricity/rto/fuel-type-data')

2026-03-07 15:48:18,052 - INFO - EIAClient initialized.
2026-03-07 15:48:18,054 - INFO - Directly accessing data endpoint metadata for: electricity/rto/fuel-type-data
2026-03-07 15:48:18,055 - INFO - Fetching metadata for route: electricity/rto/fuel-type-data


https://www.eia.gov/opendata/browser/electricity/rto/region-sub-ba-data?frequency=hourly&data=value;&sortColumn=period;&sortDirection=desc;

## Filtering options

In [2]:
route.get?

Signature:
route.get(
    data_columns: Optional[List[str]] = None,
    facets: Optional[Dict[str, Union[str, List[str]]]] = None,
    frequency: Optional[str] = None,
    start: Optional[str] = None,
    end: Optional[str] = None,
    sort: Optional[List[Dict[str, str]]] = None,
    length: Optional[int] = None,
    offset: Optional[int] = None,
    output_format: Optional[Literal['json', 'xml']] = 'json',
    paginate: bool = True,
) -> pandas.core.frame.DataFrame
Docstring:
Retrieves data from this endpoint, stores it and metadata internally,
and returns the data as a pandas DataFrame. Handles pagination automatically by default.

Args:
    data_columns: List of data column IDs to retrieve. If None, all available columns are fetched.
    facets: Dictionary of facet filters, e.g., {'stateid': 'CA', 'sectorid': ['RES', 'COM']}
    frequency: Data frequency ID (e.g., 'daily', 'monthly')
    start: Start date/period
    end: End date/period
    sort: List of sort specifications
    leng

### Facets: column values

In [3]:
route.facets

FacetContainer(facets=[respondent, fueltype])

In [4]:
facet_options = {}

for name, facet in route.facets.items():
    values = facet.get_values()
    options = []
    for meta in values:
        options.append(meta.__dict__)
    
    facet_options[name] = options

2026-03-07 15:48:19,570 - INFO - Fetching facet values for facet 'respondent' in route: electricity/rto/fuel-type-data
2026-03-07 15:48:29,368 - INFO - Fetching facet values for facet 'fueltype' in route: electricity/rto/fuel-type-data


In [5]:
import yaml

with open("facet_options.yml", "w") as f:
    yaml.dump(facet_options, f, sort_keys=False)

### Frequencies: datetime intervals

In [6]:
pd.DataFrame(route.frequencies)

,id,description,query,format
0,hourly,One data point for each hour in UTC time.,H,"YYYY-MM-DD""T""HH24"
1,local-hourly,One data point for each hour in local time.,LH,"YYYY-MM-DD""T""HH24TZH"


## Query data

In [7]:
df = route.get(
    data_columns=['value'],
    facets={'respondent': 'IPCO'},
    frequency='local-hourly',
    start='2024-01-01T00-05',
    end='2025-01-01T00-05',
    sort=[
        {'column': 'period','direction': 'desc'},
        {'column': 'fueltype','direction': 'asc'}
    ]
)

2026-03-07 15:48:39,032 - INFO - Fetching data for route: electricity/rto/fuel-type-data - Offset: 0, Length: All (Pagination)
2026-03-07 15:48:39,041 - INFO - Fetching data for route: electricity/rto/fuel-type-data/data


2026-03-07 15:49:02,716 - INFO - Received 5000 rows.
2026-03-07 15:49:02,716 - INFO - Fetching data for route: electricity/rto/fuel-type-data - Offset: 5000, Length: All (Pagination)
2026-03-07 15:49:02,717 - INFO - Fetching data for route: electricity/rto/fuel-type-data/data
2026-03-07 15:49:25,027 - INFO - Received 5000 rows.
2026-03-07 15:49:25,028 - INFO - Fetching data for route: electricity/rto/fuel-type-data - Offset: 10000, Length: All (Pagination)
2026-03-07 15:49:25,029 - INFO - Fetching data for route: electricity/rto/fuel-type-data/data
2026-03-07 15:49:51,234 - INFO - Received 5000 rows.
2026-03-07 15:49:51,234 - INFO - Fetching data for route: electricity/rto/fuel-type-data - Offset: 15000, Length: All (Pagination)
2026-03-07 15:49:51,235 - INFO - Fetching data for route: electricity/rto/fuel-type-data/data
2026-03-07 15:50:12,343 - INFO - Received 5000 rows.
2026-03-07 15:50:12,343 - INFO - Fetching data for route: electricity/rto/fuel-type-data - Offset: 20000, Length: 

In [8]:
df.period

0        2024-12-31 21:00:00-08:00
1        2024-12-31 21:00:00-08:00
                   ...            
52963    2023-12-31 21:00:00-08:00
52964    2023-12-31 21:00:00-08:00
Name: period, Length: 52965, dtype: object

## Preprocess data

In [9]:
import pandas as pd

In [10]:
x = pd.to_datetime(df.period, utc=True)
x

0       2025-01-01 05:00:00+00:00
1       2025-01-01 05:00:00+00:00
                   ...           
52963   2024-01-01 05:00:00+00:00
52964   2024-01-01 05:00:00+00:00
Name: period, Length: 52965, dtype: datetime64[ns, UTC]

In [14]:
x = x.dt.tz_convert('America/Boise')
x


0       2024-12-31 22:00:00-07:00
1       2024-12-31 22:00:00-07:00
                   ...           
52963   2023-12-31 22:00:00-07:00
52964   2023-12-31 22:00:00-07:00
Name: period, Length: 52965, dtype: datetime64[ns, America/Boise]

In [15]:
df.period = x
df

,period,respondent,respondent-name,fueltype,type-name,value,value-units
0,2024-12-31 22:00:00-07:00,IPCO,Idaho Power Company,NG,Natural Gas,766,megawatthours
1,2024-12-31 22:00:00-07:00,IPCO,Idaho Power Company,OIL,Petroleum,0,megawatthours
...,...,...,...,...,...,...,...
52963,2023-12-31 22:00:00-07:00,IPCO,Idaho Power Company,WAT,Hydro,821,megawatthours
52964,2023-12-31 22:00:00-07:00,IPCO,Idaho Power Company,WND,Wind,2,megawatthours


## Export data

In [16]:
df.to_parquet('../fuel_type_data_Idaho(IPCO).parquet')